# Classical Baselines — 5-Fold Cross-Validation

Trains logistic regression, LightGBM and CatBoost on the frozen 5-fold split and reports AUC, average precision, precision@10% and recall@10%. Random forest is trained separately in `baselines_rf_local.py`, scikit-learn's implementation being CPU-only.

CatBoost replaces XGBoost: it was the strongest single model in the source ensemble (0.98479 against XGBoost's 0.98317), and downstream TreeSHAP work requires its checkpoint.

**Inputs (Add Input):** the top-88 dataset; the repo code, including `data_loader.py` and the `folds/` directory.
**Outputs:** `results_per_fold_kaggle.csv`, `oof_<model>.npy`, per-fold checkpoints.

**Protocol.** Each model trains on `folds != k` and is scored on fold `k`. Boosters early-stop on an inner slice of the training rows, never the scoring fold. Scaling is fitted on training rows only. Feature order is asserted before prediction.

**Runtime.** Approximately 6 minutes for all five folds on 4 vCPU and one T4.

## 1 · Locate the repo code

Resolves `data_loader.py` from the working directory, a cloned repository or an attached dataset. Warns when several copies are attached or the kernel holds a stale import.

In [ ]:
import sys
from pathlib import Path

CODE_DIR_OVERRIDE = None   # e.g. "/kaggle/input/datasets/jwuptr/xmvnet-code2" if several are attached

attached = ([p.parent for p in sorted(Path("/kaggle/input").glob("**/data_loader.py"))]
            if Path("/kaggle/input").exists() else [])
CANDIDATES = ([Path(CODE_DIR_OVERRIDE)] if CODE_DIR_OVERRIDE
              else [Path.cwd(), Path("/kaggle/working/Interpretable-Churn-Predictor"), *attached])

CODE_DIR = next((p for p in CANDIDATES if (p / "data_loader.py").exists()), None)
if CODE_DIR is None:
    raise FileNotFoundError(
        "data_loader.py not found. Attach the repo as a dataset, set CODE_DIR_OVERRIDE, or run:\n"
        "  !git clone <repo-url> /kaggle/working/Interpretable-Churn-Predictor"
    )
sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import sklearn
import data_loader as dl

if len(attached) > 1:
    print("WARNING: several code copies are attached -- set CODE_DIR_OVERRIDE if this picks the wrong one")
    for p in attached:
        print("   ", p)
if Path(dl.__file__).resolve().parent != Path(CODE_DIR).resolve():
    print(f"WARNING: data_loader was already imported from {dl.__file__}"
          " -- restart the kernel before trusting this run")

print("code:", CODE_DIR)
print("data_loader:", dl.__file__)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2 · Configuration

Run settings; the only cell requiring edits.

In [ ]:
SEED = 42
RUN_TAG = "kaggle"                     # names the output files; the local forest run uses "local_rf"

MODELS = ["logreg", "lgbm", "catboost"]
DROP_VIEWS = []                        # ["recency"] -> ablation A7
SAMPLE_ROWS = None                     # e.g. 50_000 while debugging; None = all 595,000 rows

LGB_LR = 0.05                          # 0.005 reproduces the competition run, much slower
DEVICE = "cuda"                        # CatBoost only; use "cpu" without a GPU
SAVE_CHECKPOINTS = True                # Tausif needs the LightGBM / CatBoost models for TreeSHAP

TOP_FRACTION = 0.10                    # retention budget: the top 10% of customers
EARLY_STOP_FRACTION = 0.10             # inner slice of training rows, for early stopping
MAX_ROUNDS, EARLY_STOP_ROUNDS = 10_000, 200

OUT_DIR = (Path("/kaggle/working/outputs/baselines") if Path("/kaggle/working").exists()
           else CODE_DIR / "outputs" / "baselines")
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR)

## 3 · Model parameters

Hyperparameters per model, matching the source ensemble at a faster learning rate. No class reweighting is applied: churn is 12.68% and AUC is rank-based. LightGBM runs on CPU, its CUDA build being unstable on this feature matrix.

In [ ]:
LGB_PARAMS = dict(objective="binary", metric="auc", boosting_type="gbdt", max_bin=255,
                  learning_rate=LGB_LR, num_leaves=127, max_depth=8, min_child_samples=100,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                  reg_alpha=0.2, reg_lambda=1.5, verbose=-1, seed=SEED, force_col_wise=True)

CAT_PARAMS = dict(loss_function="Logloss", eval_metric="AUC", depth=7, learning_rate=0.05,
                  n_estimators=10_000, l2_leaf_reg=4.0, border_count=254, random_seed=SEED,
                  od_type="Iter", od_wait=EARLY_STOP_ROUNDS, verbose=0,
                  task_type="GPU" if DEVICE == "cuda" else "CPU")

RUN_PARAMS = {"lgbm": LGB_PARAMS, "catboost": CAT_PARAMS}

for lib in ["lightgbm", "catboost"]:
    try:
        __import__(lib)
        print(f"{lib}: available")
    except ImportError:
        print(f"{lib}: MISSING  (pip install {lib})")

## 4 · Data and folds

Loads the cached feature matrix and the frozen 5-fold split. Verifies row order against the split's recorded account IDs and the column list against `frozen_folds_5.checks.json`.

In [ ]:
import json

train = dl.load_train(n_rows=SAMPLE_ROWS)


def find_file(name):
    """Works for the repo layout, a flat dataset upload, or any attached dataset."""
    return next((Path(p) for p in [
        CODE_DIR / "folds" / name,
        CODE_DIR / name,
        *(sorted(Path("/kaggle/input").glob(f"**/{name}")) if Path("/kaggle/input").exists() else []),
    ] if Path(p).exists()), None)


FOLDS_FILE = find_file("frozen_folds_5.npy")
if FOLDS_FILE is None:
    raise FileNotFoundError("frozen_folds_5.npy not found -- attach the repo or the folds dataset")

folds = dl.make_folds(train.y, ids=train.ids, frozen_path=FOLDS_FILE)
y = train.y.astype(np.int32)
N_SPLITS = int(folds.max()) + 1

CHECKS_FILE = find_file("frozen_folds_5.checks.json")
if CHECKS_FILE is None:
    print("FEATURE_COLS check skipped: frozen_folds_5.checks.json is not attached")
else:
    expected = json.loads(CHECKS_FILE.read_text())["matrix"]["feature_cols"]
    assert train.feature_cols == expected, "FEATURE_COLS differ from the verified list"
    print("FEATURE_COLS match the verified list:", CHECKS_FILE)

print(f"{len(y):,} rows | {train.n_features} columns | {N_SPLITS} folds | churn {y.mean():.5f}")
print(train.summary().to_string(index=False))

## 5 · Metrics

AUC, average precision, and precision/recall within the top 10% of predicted risk. Recall@10% is bounded at 78.9%: 11,900 customers selected against 15,087 churners per fold.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

METRIC_NAMES = ["auc", "ap", "p_at_10", "r_at_10"]


def evaluate(y_true, scores):
    k = max(1, int(round(TOP_FRACTION * len(y_true))))
    top_k = np.argpartition(-scores, k - 1)[:k]
    hits = int(y_true[top_k].sum())
    return {
        "auc": float(roc_auc_score(y_true, scores)),
        "ap": float(average_precision_score(y_true, scores)),
        "p_at_10": hits / k,
        "r_at_10": hits / max(1, int(y_true.sum())),
    }

## 6 · Columns and scaling

Applies `DROP_VIEWS` through the loader's view groups (`["recency"]` gives ablation A7). Standardisation is fitted on training rows only.

In [ ]:
def select_columns(train, drop_views):
    keep = np.ones(train.n_features, dtype=bool)
    for view in drop_views:
        if view not in train.view_groups:
            raise KeyError(f"unknown view {view!r}; available: {list(train.view_groups)}")
        keep[train.view_groups[view]] = False
    idx = np.where(keep)[0]
    return idx, [train.feature_cols[i] for i in idx]


def standardize(Xtr, *others):
    mean = Xtr.mean(axis=0, dtype=np.float64)
    std = Xtr.std(axis=0, dtype=np.float64)
    std[std < 1e-6] = 1.0
    return [((X - mean) / std).astype(np.float32) for X in (Xtr, *others)]


COL_IDX, COLS = select_columns(train, DROP_VIEWS)
print(f"{len(COLS)} feature columns"
      + (f"  (dropped views: {', '.join(DROP_VIEWS)})" if DROP_VIEWS else ""))

## 7 · Models

One trainer per model, each returning `(model, scores, best_iteration)`. Boosters early-stop on an inner 10% slice of the training rows. Feature order is asserted against `COLS` before prediction.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _inner_split(y_tr):
    return train_test_split(np.arange(len(y_tr)), test_size=EARLY_STOP_FRACTION,
                            stratify=y_tr, random_state=SEED)


def fit_logreg(Xtr, ytr, Xva, yva):
    Xtr_s, Xva_s = standardize(Xtr, Xva)
    model = LogisticRegression(C=1.0, max_iter=2000, n_jobs=-1, random_state=SEED)
    model.fit(Xtr_s, ytr)
    assert model.n_features_in_ == len(COLS)
    return model, model.predict_proba(Xva_s)[:, 1], None


def fit_lgbm(Xtr, ytr, Xva, yva):
    import lightgbm as lgb

    fit_idx, es_idx = _inner_split(ytr)
    d_fit = lgb.Dataset(Xtr[fit_idx], ytr[fit_idx], feature_name=COLS)
    d_es = lgb.Dataset(Xtr[es_idx], ytr[es_idx], feature_name=COLS, reference=d_fit)
    model = lgb.train(LGB_PARAMS, d_fit, num_boost_round=MAX_ROUNDS, valid_sets=[d_es],
                      callbacks=[lgb.early_stopping(EARLY_STOP_ROUNDS, verbose=False)])
    assert model.feature_name() == COLS, "LightGBM feature order drifted"
    return model, model.predict(Xva, num_iteration=model.best_iteration), model.best_iteration


def fit_catboost(Xtr, ytr, Xva, yva):
    from catboost import CatBoostClassifier, Pool

    fit_idx, es_idx = _inner_split(ytr)
    model = CatBoostClassifier(**CAT_PARAMS)
    model.fit(Pool(Xtr[fit_idx], ytr[fit_idx], feature_names=COLS),
              eval_set=Pool(Xtr[es_idx], ytr[es_idx], feature_names=COLS), use_best_model=True)
    assert list(model.feature_names_) == COLS, "CatBoost feature order drifted"
    return model, model.predict_proba(Xva)[:, 1], model.get_best_iteration()


TRAINERS = {"logreg": fit_logreg, "lgbm": fit_lgbm, "catboost": fit_catboost}
print("trainers:", list(TRAINERS))

## 8 · Checkpoints

Writes each fold's model in its library's native format.

In [ ]:
def save_model(model, name, fold):
    if model is None or not SAVE_CHECKPOINTS:
        return None
    models_dir = OUT_DIR / "models"
    if name == "lgbm":
        path = models_dir / f"lgbm_fold{fold}.txt"
        model.save_model(str(path), num_iteration=model.best_iteration)
    elif name == "catboost":
        path = models_dir / f"catboost_fold{fold}.cbm"
        model.save_model(str(path))
    else:
        import joblib
        path = models_dir / f"{name}_fold{fold}.joblib"
        joblib.dump(model, path, compress=3)
    return path.name

## 9 · Fold loop

Trains each model on `folds != k`, scores fold `k`, and records the metrics. Out-of-fold predictions accumulate into one vector per model.

In [ ]:
import time

rows, oof = [], {m: np.full(len(y), np.nan) for m in MODELS}

for fold in range(N_SPLITS):
    tr_idx = np.where(folds != fold)[0]
    va_idx = np.where(folds == fold)[0]
    Xtr = np.asarray(train.X[tr_idx])[:, COL_IDX]
    Xva = np.asarray(train.X[va_idx])[:, COL_IDX]
    ytr, yva = y[tr_idx], y[va_idx]

    for name in MODELS:
        t0 = time.perf_counter()
        model, scores, best_iter = TRAINERS[name](Xtr, ytr, Xva, yva)
        seconds = time.perf_counter() - t0

        oof[name][va_idx] = scores
        metrics = evaluate(yva, scores)
        rows.append({"model": name, "fold": fold, **metrics, "seconds": round(seconds, 1),
                     "best_iteration": best_iter, "n_features": len(COLS),
                     "checkpoint": save_model(model, name, fold)})
        print(f"fold {fold} {name:<9} auc {metrics['auc']:.5f}  ap {metrics['ap']:.5f}  "
              f"p@10 {metrics['p_at_10']:.4f}  r@10 {metrics['r_at_10']:.4f}  {seconds:7.1f}s")

    del Xtr, Xva

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / f"results_per_fold_{RUN_TAG}.csv", index=False)
for name, preds in oof.items():
    np.save(OUT_DIR / f"oof_{name}.npy", preds)

display(results)

## 10 · Summary and run record

Mean ± standard deviation across folds. `run_<tag>.json` records the fold file, seed, parameters and library versions.

In [ ]:
import json

agg = results.groupby("model")[METRIC_NAMES].agg(["mean", "std"])
table = pd.DataFrame({m: agg[(m, "mean")].map("{:.5f}".format) + " ± "
                      + agg[(m, "std")].map("{:.5f}".format) for m in METRIC_NAMES})
table["total_seconds"] = results.groupby("model")["seconds"].sum().round(0)

manifest_path = Path(FOLDS_FILE).with_suffix(".json")
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

(OUT_DIR / f"run_{RUN_TAG}.json").write_text(json.dumps({
    "run_tag": RUN_TAG, "models": MODELS, "drop_views": DROP_VIEWS, "sample_rows": SAMPLE_ROWS,
    "seed": SEED, "n_features": len(COLS), "params": RUN_PARAMS,
    "folds": {"file": Path(FOLDS_FILE).name, "seed": manifest.get("seed"),
              "ids_sha256": manifest.get("ids_sha256")},
    "versions": {"numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__},
    "summary": {m: {k: {"mean": float(agg[(m, "mean")][k]), "std": float(agg[(m, "std")][k])}
                    for k in agg.index} for m in METRIC_NAMES},
}, indent=2))

print(table.to_string())
print("\nsaved ->", OUT_DIR)

## 11 · Packaging

Bundles checkpoints, result tables and out-of-fold predictions into one archive. Save Version before downloading; unsaved files in `/kaggle/working` are discarded.

`logreg_fold<k>.joblib` has no saved scaler and cannot score data on its own; use `train_full_models.ipynb` for a usable logistic-regression checkpoint.

In [ ]:
import zipfile

zip_path = OUT_DIR.parent / "baseline_fold_checkpoints.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted((OUT_DIR / "models").rglob("*")):
        if p.is_file():
            z.write(p, f"models/{p.name}")
    for pattern in ("results_per_fold_*.csv", "run_*.json", "oof_*.npy"):
        for p in sorted(OUT_DIR.glob(pattern)):
            z.write(p, p.name)

print(f"{zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)\n")
for info in sorted(zipfile.ZipFile(zip_path).infolist(), key=lambda i: i.filename):
    print(f"  {info.filename:<34} {info.file_size / 1e6:7.2f} MB")
print("\nSave Version, then download from the finished version's Output tab.")